# 01 · Chạy thử BGE-M3 trên 10 văn bản (Tuần 1 · TV2)

**Mục tiêu:** xác nhận môi trường chạy được trọn chuỗi *văn bản → chia đoạn → nhúng BGE-M3 → Qdrant → truy vấn*, và đo thời gian, VRAM để ước lượng cho toàn bộ kho ~37 văn bản.

**Dữ liệu:** 10 nghị quyết của UBTVQH về sắp xếp ĐVHC cấp xã (liệt kê trong `data/raw/manifest.csv`), tải từ Cổng văn bản Chính phủ. Bản gốc là PDF scan nên văn bản ở đây là **bản OCR nháp chưa soát lỗi** (`data/raw/ocr_draft/`). Vì vậy notebook này chỉ kiểm tra hạ tầng; các con số truy xuất bên dưới không dùng cho bài báo.

Chuẩn bị: `pip install -e ".[rag,ocr,dev]"` rồi `python scripts/fetch_raw.py`.

In [1]:
import csv, os, sys, time
from pathlib import Path
from statistics import mean, median

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import torch
print("Python", sys.version.split()[0], "· torch", torch.__version__)
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"GPU: {torch.cuda.get_device_name(0)} · {total:.1f} GB")
else:
    print("Không có GPU — chạy trên CPU")

Python 3.13.11 · torch 2.6.0+cu124
GPU: NVIDIA GeForce GTX 1660 Ti with Max-Q Design · 6.0 GB


## 1. Nạp văn bản

In [2]:
import pandas as pd
from rag.chunkers import fixed_chunks, legal_chunks

with open("data/raw/manifest.csv", encoding="utf-8", newline="") as f:
    manifest = list(csv.DictReader(f))

docs = {}
for row in manifest:
    path = Path("data/raw") / row["file"]
    if path.exists():
        docs[row["doc_id"]] = path.read_text(encoding="utf-8")

province = {r["doc_id"]: r["province"] for r in manifest}
pd.DataFrame(
    [{"doc_id": d, "tỉnh/thành": province[d], "ký tự": len(t), "từ": len(t.split())} for d, t in docs.items()]
)

,doc_id,tỉnh/thành,ký tự,từ
0,NQ-1657-2025-UBTVQH15,Cao Bằng,11360,2625
1,NQ-1663-2025-UBTVQH15,Đồng Tháp,19126,4452
2,NQ-1665-2025-UBTVQH15,Hà Tĩnh,13213,3071
3,NQ-1667-2025-UBTVQH15,Khánh Hòa,13391,3061
4,NQ-1668-2025-UBTVQH15,Cần Thơ,17655,4069
5,NQ-1669-2025-UBTVQH15,Hải Phòng,27636,6224
6,NQ-1670-2025-UBTVQH15,Lai Châu,8381,1961
7,NQ-1671-2025-UBTVQH15,Lâm Đồng,20521,4835
8,NQ-1672-2025-UBTVQH15,Lạng Sơn,12987,3014
9,NQ-1678-2025-UBTVQH15,Nghệ An,21221,4900


## 2. Chia đoạn: cố định 512/64 và theo Điều/Khoản

In [3]:
chunks = {
    "fixed": [c for d, t in docs.items() for c in fixed_chunks(d, t)],
    "legal": [c for d, t in docs.items() for c in legal_chunks(d, t)],
}

def length_stats(cs):
    n = [len(c.text.split()) for c in cs]
    return {"số đoạn": len(n), "từ TB": round(mean(n)), "trung vị": round(median(n)), "dài nhất": max(n)}

chunk_stats = pd.DataFrame({k: length_stats(v) for k, v in chunks.items()}).T
chunk_stats

,số đoạn,từ TB,trung vị,dài nhất
fixed,87,496,512,512
legal,780,60,50,329


In [4]:
# Số Khoản mà bộ chia đoạn nhận ra ở mỗi văn bản. Lỗi OCR ở số thứ tự ("2 Sắp xếp" thay vì "2. Sắp xếp")
# làm Khoản bị gộp vào Khoản trước, nên con số này thấp hơn số đơn vị thực tế.
per_doc = pd.Series([c.doc_id for c in chunks["legal"]]).value_counts().sort_index()
pd.DataFrame({"tỉnh/thành": pd.Series(province), "đoạn theo Điều/Khoản": per_doc}).dropna()

,tỉnh/thành,đoạn theo Điều/Khoản
NQ-1657-2025-UBTVQH15,Cao Bằng,57
NQ-1663-2025-UBTVQH15,Đồng Tháp,94
NQ-1665-2025-UBTVQH15,Hà Tĩnh,66
NQ-1667-2025-UBTVQH15,Khánh Hòa,62
NQ-1668-2025-UBTVQH15,Cần Thơ,92
NQ-1669-2025-UBTVQH15,Hải Phòng,102
NQ-1670-2025-UBTVQH15,Lai Châu,38
NQ-1671-2025-UBTVQH15,Lâm Đồng,115
NQ-1672-2025-UBTVQH15,Lạng Sơn,61
NQ-1678-2025-UBTVQH15,Nghệ An,93


## 3. Nhúng bằng BGE-M3

In [5]:
from FlagEmbedding import BGEM3FlagModel

use_gpu = torch.cuda.is_available()
t0 = time.time()
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=use_gpu)
load_s = time.time() - t0
print(f"Nạp mô hình: {load_s:.1f} s")

Nạp mô hình: 5.7 s


In [6]:
MAX_LEN = 1024   # token BGE-M3; đoạn dài hơn bị cắt

def embed(cs, batch_size=8):
    texts = [c.text for c in cs]
    n_tok = [len(ids) for ids in model.tokenizer(texts, add_special_tokens=True)["input_ids"]]
    if use_gpu:
        torch.cuda.reset_peak_memory_stats()
    t = time.time()
    vecs = model.encode(texts, batch_size=batch_size, max_length=MAX_LEN)["dense_vecs"]
    dt = time.time() - t
    return vecs, {
        "số đoạn": len(cs),
        "số chiều": vecs.shape[1],
        "giây": round(dt, 1),
        "đoạn/giây": round(len(cs) / dt, 1),
        "token TB": round(mean(n_tok)),
        "đoạn bị cắt (> MAX_LEN)": sum(n > MAX_LEN for n in n_tok),
        "VRAM đỉnh (GB)": round(torch.cuda.max_memory_allocated() / 2**30, 2) if use_gpu else None,
    }

vectors, embed_stats = {}, {}
for name, cs in chunks.items():
    vectors[name], embed_stats[name] = embed(cs)
embed_stats = pd.DataFrame(embed_stats).T
embed_stats

,số đoạn,số chiều,giây,đoạn/giây,token TB,đoạn bị cắt (> MAX_LEN),VRAM đỉnh (GB)
fixed,87.0,1024.0,85.2,1.0,653.0,0.0,1.20
legal,780.0,1024.0,102.2,7.6,79.0,0.0,1.14


## 4. Ghi vào Qdrant và truy vấn thử

Dùng Qdrant dạng nhúng (`QDRANT_PATH`, không cần Docker). Collection thử nằm riêng trong `qdrant_storage/trial`, không đụng tới chỉ mục chính của Tuần 2.

In [7]:
os.environ["QDRANT_PATH"] = "qdrant_storage/trial"
from qdrant_client.models import Distance, PointStruct, VectorParams
from rag.vector_store import get_qdrant

qd = get_qdrant()
for name, cs in chunks.items():
    coll = f"trial_{name}"
    if qd.collection_exists(coll):
        qd.delete_collection(coll)
    qd.create_collection(coll, vectors_config=VectorParams(size=vectors[name].shape[1], distance=Distance.COSINE))
    qd.upsert(coll, points=[
        PointStruct(id=i, vector=vectors[name][i].tolist(),
                    payload={"chunk_id": c.chunk_id, "doc_id": c.doc_id, "text": c.text, **c.metadata})
        for i, c in enumerate(cs)
    ])
    print(coll, qd.count(coll).count, "điểm")

trial_fixed 87 điểm


trial_legal 780 điểm


In [8]:
# Truy vấn kiểm tra nhanh: mỗi câu gắn với văn bản mà đáp án phải nằm trong đó.
queries = [
    ("Xã Lạc Dương của tỉnh Lâm Đồng được thành lập từ những xã nào?", "NQ-1671-2025-UBTVQH15"),
    ("Thị trấn Thạnh Mỹ và xã Đạ Ròn sau sắp xếp thành xã nào?", "NQ-1671-2025-UBTVQH15"),
    ("Sắp xếp các đơn vị hành chính cấp xã của tỉnh Cao Bằng", "NQ-1657-2025-UBTVQH15"),
    ("Sau sắp xếp, thành phố Hải Phòng có bao nhiêu đơn vị hành chính cấp xã?", "NQ-1669-2025-UBTVQH15"),
    ("Sau sắp xếp, tỉnh Nghệ An có bao nhiêu xã và phường?", "NQ-1678-2025-UBTVQH15"),
    ("Các đơn vị hành chính cấp xã của tỉnh Lai Châu không thực hiện sắp xếp", "NQ-1670-2025-UBTVQH15"),
]
q_vecs = model.encode([q for q, _ in queries], max_length=256)["dense_vecs"]

rows = []
for (q, want), v in zip(queries, q_vecs):
    for name in chunks:
        hits = qd.query_points(f"trial_{name}", query=v.tolist(), limit=3).points
        rows.append({
            "câu hỏi": q, "chia đoạn": name,
            "đúng văn bản trong top-3": any(h.payload["doc_id"] == want for h in hits),
            "top-1": hits[0].payload["chunk_id"], "điểm": round(hits[0].score, 3),
        })
search = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
search

,câu hỏi,chia đoạn,đúng văn bản trong top-3,top-1,điểm
0,Xã Lạc Dương của tỉnh Lâm Đồng được thành lập từ những xã nào?,fixed,True,NQ-1671-2025-UBTVQH15#fixed0,0.504
1,Xã Lạc Dương của tỉnh Lâm Đồng được thành lập từ những xã nào?,legal,True,NQ-1671-2025-UBTVQH15#Art1.1,0.607
2,Thị trấn Thạnh Mỹ và xã Đạ Ròn sau sắp xếp thành xã nào?,fixed,True,NQ-1663-2025-UBTVQH15#fixed2,0.548
3,Thị trấn Thạnh Mỹ và xã Đạ Ròn sau sắp xếp thành xã nào?,legal,True,NQ-1671-2025-UBTVQH15#Art1.1,0.526
4,Sắp xếp các đơn vị hành chính cấp xã của tỉnh Cao Bằng,fixed,True,NQ-1657-2025-UBTVQH15#fixed4,0.660
5,Sắp xếp các đơn vị hành chính cấp xã của tỉnh Cao Bằng,legal,True,NQ-1657-2025-UBTVQH15#Art1.57,0.746
6,"Sau sắp xếp, thành phố Hải Phòng có bao nhiêu đơn vị hành chính cấp xã?",fixed,True,NQ-1669-2025-UBTVQH15#fixed12,0.733
7,"Sau sắp xếp, thành phố Hải Phòng có bao nhiêu đơn vị hành chính cấp xã?",legal,True,NQ-1669-2025-UBTVQH15#Art1.115,0.759
8,"Sau sắp xếp, tỉnh Nghệ An có bao nhiêu xã và phường?",fixed,True,NQ-1678-2025-UBTVQH15#fixed9,0.708
9,"Sau sắp xếp, tỉnh Nghệ An có bao nhiêu xã và phường?",legal,True,NQ-1678-2025-UBTVQH15#Art1.122,0.685


In [9]:
# Xem nội dung top-3 của câu đầu tiên trên chỉ mục theo Điều/Khoản.
for h in qd.query_points("trial_legal", query=q_vecs[0].tolist(), limit=3).points:
    print(f"[{h.score:.3f}] {h.payload['chunk_id']}\n{h.payload['text'][:300]}\n")

[0.607] NQ-1671-2025-UBTVQH15#Art1.1
Điều 1. Sắp xếp các đơn vị hành chính cấp xã của tỉnh Lâm Đồng
1. Sắp xếp toàn bộ diện tích tự nhiên, quy mô dân số của các xã Đạ Sar, Đạ
Nhim và Đạ Chais thành xã mới có tên gọi là xã Lạc Dương:
2 Sắp xếp toàn bộ diện tích tự nhiên, quy mô dân số của thị trấn Thạnh Mỹ,
xã Đạ Ròn và xã Tu Tra thành 

[0.530] NQ-1671-2025-UBTVQH15#Art1.29
Điều 1. Sắp xếp các đơn vị hành chính cấp xã của tỉnh Lâm Đồng
29. Sắp xếp toàn bộ diện tích tự nhiên, quy mô dân số của các xã Lộc An,
Lộc Đức và Tân Lạc thành xã mới có tên gọi là xã Bảo Lâm 2.

[0.519] NQ-1671-2025-UBTVQH15#Art1.71
Điều 1. Sắp xếp các đơn vị hành chính cấp xã của tỉnh Lâm Đồng
71. Sắp xếp toàn bộ diện tích tự nhiên; quy mô dân số của thị trấn Lạc Tánh;
xã Gia An và xã Đức Thuận thành xã mới có tên gọi à xã Tánb Linh



## 5. Tóm tắt

In [10]:
hit = search.groupby("chia đoạn")["đúng văn bản trong top-3"].agg(["sum", "count"])
print(f"Văn bản: {len(docs)} · tổng {sum(len(t.split()) for t in docs.values()):,} từ")
print(f"Nạp BGE-M3: {load_s:.1f} s")
for name in chunks:
    s = embed_stats.loc[name]
    print(f"{name:5s}: {int(s['số đoạn'])} đoạn · {s['giây']} s · {s['đoạn/giây']} đoạn/s · "
          f"VRAM đỉnh {s['VRAM đỉnh (GB)']} GB · đúng văn bản trong top-3: {hit.loc[name, 'sum']}/{hit.loc[name, 'count']}")
qd.close()

Văn bản: 10 · tổng 38,212 từ
Nạp BGE-M3: 5.7 s
fixed: 87 đoạn · 85.2 s · 1.0 đoạn/s · VRAM đỉnh 1.2 GB · đúng văn bản trong top-3: 6/6
legal: 780 đoạn · 102.2 s · 7.6 đoạn/s · VRAM đỉnh 1.14 GB · đúng văn bản trong top-3: 6/6
